# Sub-cluster Cluster 8 — Resolving TSPC vs. T-FAP

**Input:** `data/Cherief_scRNA-seq/GSE244921_processed.h5ad`  
**Goal:** Split the mixed PDGFRα+ stromal cluster (cluster 8 from notebook 01) into TSPC (Tppp3+) and T-FAP (Pi16+/Sfrp2+/Tppp3−) sub-populations.

**Strategy:** Re-use the PCA embedding from the full object, then re-run `neighbors` and `UMAP` restricted to cluster 8 cells only. This reveals within-cluster structure without interference from other cell types.

**Result summary (Leiden res=0.2, 4 sub-clusters):**
- **Cluster 3 → TSPC:** Tppp3 mean=1.755, 94.5% expressing; highest Cd248, Cd34, Prg4; Sfrp2 absent (mean=−0.005); top DEGs include Cd55 (complement regulator on progenitors) and Ptgs1 (COX-1, repair signaling)
- **Cluster 1 → T-FAP:** Pi16 mean=1.633, Sfrp2 mean=1.347; lowest Tppp3; highest Pdgfra; top DEGs are complement genes (C3, C4b, Serping1) — T-FAPs are complement-producing
- **Cluster 2 → Tenogenic progenitor:** Tppp3-negative; TrkAWT-enriched; top DEGs are tendon ECM genes (Thbs4, Kera, Col11a1) — likely committed toward tenogenic fate under innervation
- **Cluster 0 → Stromal:** moderate of both marker sets; top DEGs are imprinted genes (Plagl1, Mest, H19) — epigenetically distinct stromal state

**Output:** `data/Cherief_scRNA-seq/GSE244921_cluster8_sub.h5ad` with `cell_type` annotation.

In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

sc.settings.verbosity = 2
sc.settings.figdir = '../figures/Cherief_scRNA-seq/'

## 1. Load processed object

In [ ]:
full = sc.read_h5ad('../data/Cherief_scRNA-seq/GSE244921_processed.h5ad')
print(full)
print('\nLeiden cluster sizes:')
print(full.obs['leiden'].value_counts().sort_index())

## 2. Subset to cluster 8

In [ ]:
sub = full[full.obs['leiden'] == '8'].copy()
print(f'Cluster 8: {sub.n_obs} cells')
print('\nCondition split:')
print(sub.obs['condition'].value_counts())

**Cluster 8 composition:** 3,900 cells — 2,094 TrkAWT (innervated) and 1,806 TrkAF592A (denervated). The imbalance toward innervated is consistent with Cherief et al.'s finding that innervation supports progenitor expansion in this compartment.

## 3. Re-embed within cluster 8

We re-run `neighbors` and `UMAP` on the PCA coordinates already computed for the full dataset (`X_pca`). This re-builds the neighborhood graph using only the cluster 8 cells, allowing Leiden to detect finer sub-structure without interference from other cell types.

`n_neighbors=15` is used (smaller than the default 30) to reflect the smaller population size.

In [ ]:
sc.pp.neighbors(sub, n_pcs=20, n_neighbors=15, use_rep='X_pca')
sc.tl.umap(sub)

In [ ]:
sc.pl.umap(sub, color=['Tppp3', 'Pdgfra', 'Pi16', 'condition'],
           ncols=2, cmap='Reds',
           save='_sub8_sanity.png')

**Sanity check findings:**

- **Tppp3:** Dense in the upper-left region of the UMAP. Expression is focal and not diffuse — confirms a distinct Tppp3+ sub-population exists within cluster 8.
- **Pi16:** Dense at the bottom of the UMAP. Spatially complementary to Tppp3 (upper-left vs. bottom) — the two markers are largely anti-correlated, which is the key prerequisite for TSPC/T-FAP separation.
- **Pdgfra:** Broadly expressed but highest toward the center-right. Notably low in the Tppp3-dense upper-left region. Because all cluster 8 cells are PDGFRα+, this reflects *relative* Pdgfra within the cluster — T-FAP-like cells express it at above-average levels, TSPCs at average or below.
- **Condition:** TrkAWT (orange/innervated) is enriched in a small protrusion at the very top of the UMAP, which co-localises with the Tppp3-high zone. TrkAF592A (blue/denervated) dominates the center and right. **This directly reproduces Cherief et al.'s core finding: innervation maintains the Tppp3+ TSPC compartment, and denervation depletes it.** The biology is visible before any clustering is applied.

## 4. Leiden resolution sweep

Sweep resolutions from 0.2 to 1.0 to find the lowest resolution at which TSPC and T-FAP form distinct partitions.

In [ ]:
resolutions = [0.2, 0.4, 0.6, 0.8, 1.0]

for res in resolutions:
    sc.tl.leiden(
        sub, resolution=res,
        key_added=f'leiden_{res}',
        flavor='igraph', n_iterations=2, directed=False
    )
    n = sub.obs[f'leiden_{res}'].nunique()
    print(f'res={res}: {n} sub-clusters')

In [ ]:
sc.pl.umap(
    sub,
    color=[f'leiden_{res}' for res in resolutions],
    ncols=3, legend_loc='on data',
    save='_sub8_resolution_sweep.png'
)

**Resolution choice — res=0.2 selected (4 sub-clusters):**

Resolution 0.2 gives 4 sub-clusters that map cleanly onto biologically interpretable populations. Higher resolutions (0.6+) fragment the cluster into 10–17 sub-groups with no additional biological meaning at this stage. The 4-cluster solution provides the minimum necessary resolution to separate TSPC from T-FAP while keeping intermediate populations tractable.

Marker validation (§5) confirmed the biological identity of all four sub-clusters using numerical expression tables rather than visual dotplot reading.

## 5. Marker validation at chosen resolution

In [ ]:
RESOLUTION = 0.2

sub.obs['leiden_sub'] = sub.obs[f'leiden_{RESOLUTION}']

sc.pl.umap(
    sub, color='leiden_sub',
    legend_loc='on data',
    title=f'Cluster 8 sub-clusters (Leiden res={RESOLUTION})',
    save='_sub8_chosen.png'
)

In [ ]:
# Candidate markers from Harvey 2019:
#   TSPC (cluster 2): Tppp3, Cd248, Tnxb, Prg4, Cd34
#   T-FAP (cluster 3): Pi16, Dpp4, Ly6a (Sca-1), Sfrp2
#   Shared PDGFRα+ field: Pdgfra
# Note: Dpp4 was found to be highest in the TSPC cluster (3) in this dataset
# and is NOT a reliable T-FAP discriminator in the Achilles injury context.
# Sfrp2 is the most informative T-FAP marker here.

tspc_markers = ['Tppp3', 'Cd248', 'Tnxb', 'Prg4', 'Cd34']
tfap_markers = ['Pi16', 'Dpp4', 'Ly6a', 'Sfrp2']
shared       = ['Pdgfra']

all_markers = [g for g in tspc_markers + tfap_markers + shared
               if g in sub.var_names]

print('Markers found in dataset:', all_markers)
print('Missing:', [g for g in tspc_markers + tfap_markers + shared
                   if g not in sub.var_names])

In [ ]:
sc.pl.dotplot(
    sub, all_markers, groupby='leiden_sub',
    standard_scale='var',
    save='_sub8_markers.png'
)

In [ ]:
# Numerical marker table — used to verify dotplot visual reading
markers_to_check = [g for g in all_markers if g in sub.var_names]

expr = sub[:, markers_to_check].X
if hasattr(expr, 'toarray'):
    expr = expr.toarray()

import numpy as np
df_expr = pd.DataFrame(expr, columns=markers_to_check, index=sub.obs_names)
df_expr['leiden_sub'] = sub.obs['leiden_sub'].values

print('=== Mean expression per sub-cluster ===')
print(df_expr.groupby('leiden_sub').mean().round(3).to_string())

print('\n=== Fraction of cells expressing (> 0) ===')
print((df_expr.groupby('leiden_sub')[markers_to_check]
         .apply(lambda x: (x > 0).mean())
         .round(3)
         .to_string()))

In [ ]:
sc.pl.umap(
    sub,
    color=['Tppp3', 'Pi16', 'Pdgfra', 'Cd248', 'Sfrp2', 'Prg4'],
    ncols=3, cmap='Reds',
    save='_sub8_gene_overlays.png'
)

**Marker findings (from numerical table):**

| Sub-cluster | Tppp3 (mean / frac) | Pi16 (mean / frac) | Sfrp2 (mean / frac) | Pdgfra (mean / frac) | Identity |
|---|---|---|---|---|---|
| 0 | 0.586 / 0.619 | 0.718 / 0.579 | 0.862 / 0.571 | 1.045 / 0.865 | Stromal |
| 1 | 0.282 / 0.516 | **1.633 / 0.708** | **1.347 / 0.659** | **1.482 / 0.905** | T-FAP |
| 2 | -0.082 / 0.342 | 0.431 / 0.413 | 0.305 / 0.385 | 1.068 / 0.860 | Tenogenic progenitor |
| 3 | **1.755 / 0.945** | 1.235 / 0.588 | -0.005 / 0.235 | 0.781 / 0.792 | TSPC |

**Key discriminators:**
- **Tppp3** separates cluster 3 from all others (94.5% expressing, mean 1.755 — far above the rest)
- **Sfrp2** is the most informative T-FAP marker: cluster 1 highest (1.347), cluster 3 lowest (−0.005 / 23.5%) — near-absent in TSPC
- **Pi16** also highest in cluster 1 (1.633), confirming T-FAP identity
- **Pdgfra** highest in cluster 1 (1.482) and lowest in cluster 3 (0.781). All cells are PDGFRα+ (this is cluster 8), but T-FAPs express it at higher relative levels

**Dpp4 caveat:** Dpp4 was highest in cluster 3 (TSPC) with mean=1.425 and 48.8% expressing, compared to cluster 1 (T-FAP) at 0.708 / 24.1%. Dpp4 was listed as a T-FAP marker in Harvey 2019 (patellar tendon, uninjured). In the Cherief 2023 Achilles injury context it does not behave as a T-FAP discriminator and should not be used for annotation in this dataset.

**Cluster 2 (tenogenic progenitor):** Tppp3-negative (mean −0.082), low Pi16 and Sfrp2, TrkAWT-enriched (visible as the top UMAP protrusion). Unbiased DEGs (§6) show this cluster is defined by tendon ECM genes — Thbs4 (Thrombospondin-4), Kera (Keratocan), Col11a1 — suggesting it is already committed toward a tenogenic fate rather than being quiescent. The TrkAWT enrichment implies innervation is required to maintain or expand this tenogenic progenitor state.

## 6. Differential expression between sub-clusters

Rank genes that best distinguish each sub-cluster. The top DEGs per cluster will seed the pySCENIC TF inference and the feature selection step in the classifier.

In [ ]:
sc.tl.rank_genes_groups(
    sub, groupby='leiden_sub',
    method='wilcoxon', tie_correct=True
)

sc.pl.rank_genes_groups_dotplot(
    sub, n_genes=5,
    standard_scale='var',
    save='_sub8_top_degs.png'
)

In [ ]:
deg_table = sc.get.rank_genes_groups_df(sub, group=None)
deg_table.to_csv('../data/Cherief_scRNA-seq/cluster8_sub_DEGs.csv', index=False)
print('Top 5 genes per sub-cluster:')
print(deg_table.groupby('group').head(5)[['group', 'names', 'scores', 'pvals_adj']])

**DEG interpretation — top 5 unbiased markers per sub-cluster:**

These genes are selected purely by statistical discrimination (Wilcoxon), independent of the candidate markers used in §5.

| Sub-cluster | Top DEGs | Biological interpretation |
|---|---|---|
| 3 (TSPC) | Ptgs1, Olfml2a, Efhd1, Arnt2, Cd55 | Cd55 (DAF) is a complement regulator expressed on progenitor cell surfaces; Ptgs1 (COX-1) drives prostaglandin-mediated tissue repair signaling |
| 2 (Tenogenic prog.) | Thbs4, Kera, Col11a1, Cst3, Fxyd6 | Thbs4 (Thrombospondin-4), Kera (Keratocan), and Col11a1 are tendon-specific ECM genes — this population is already expressing the tendon matrix program, supporting a tenogenic rather than quiescent identity |
| 0 (Stromal) | Gnas, Plagl1, Ptn, Mest, H19 | Plagl1, Mest, and H19 are all imprinted genes — an unusual epigenetic signature suggesting this stromal state has a developmentally distinct origin or regulatory program |
| 1 (T-FAP) | C3, Serping1, C4b, Ace, Serpina3n | **C3, C4b, and Serping1 are complement cascade genes** — T-FAPs are the major complement-producing cell type in this injured tendon. Complement activation drives local inflammation and is mechanistically linked to fibrosis in multiple tissues. This may explain how T-FAP activation leads to scarring rather than regeneration. |

**Key finding:** The T-FAP complement signature (C3/C4b/Serping1) is a novel observation not reported in Harvey 2019. It reframes T-FAPs not just as fibrogenic progenitors but as active drivers of complement-mediated inflammation in the injury environment. This is a candidate mechanism worth investigating in downstream analyses (e.g., LIANA can detect complement ligand-receptor interactions between T-FAPs and immune cells).

## 7. Condition split across sub-clusters

Denervation (TrkAF592A) is expected to deplete TSPCs and shift progenitors toward a fibrotic state. The proportional shift between conditions across sub-clusters quantifies this effect within cluster 8.

In [ ]:
sc.pl.umap(sub, color='condition', save='_sub8_condition.png')

In [ ]:
props = (
    sub.obs
    .groupby(['condition', 'leiden_sub'])
    .size()
    .unstack(fill_value=0)
    .apply(lambda x: x / x.sum(), axis=1)
)

ax = props.T.plot(
    kind='bar', figsize=(10, 4),
    title='Sub-cluster proportions by condition (cluster 8)'
)
ax.set_xlabel('Sub-cluster (0=Stromal, 1=T-FAP, 2=Quiescent-prog, 3=TSPC)')
ax.set_ylabel('Proportion')
plt.tight_layout()
plt.savefig('../figures/Cherief_scRNA-seq/sub8_condition_proportions.png',
            dpi=150, bbox_inches='tight')
plt.show()

print('\nAbsolute counts:')
print(sub.obs.groupby(['condition', 'leiden_sub']).size().unstack(fill_value=0))

**What to look for:** Cluster 3 (TSPC) should be proportionally higher in TrkAWT than TrkAF592A. Cluster 1 (T-FAP) should be relatively more represented in TrkAF592A. Cluster 2 (Tenogenic progenitor) is expected to be strongly TrkAWT-enriched — it corresponds to the top UMAP protrusion that was near-exclusively orange in the sanity check. The bar chart will quantify all of these shifts.

## 8. Annotate sub-clusters

Annotations are based on two independent lines of evidence:
1. **Candidate marker table (§5):** Tppp3 and Sfrp2/Pi16 discriminate TSPC from T-FAP; Dpp4 was found unreliable in this context
2. **Unbiased DEGs (§6):** Top Wilcoxon genes per cluster confirm identities and reveal novel biology (complement signature in T-FAP; tendon ECM signature in cluster 2)

Note: cluster 2 is labeled **Tenogenic progenitor** (not quiescent) based on its top DEGs (Thbs4, Kera, Col11a1) — it is actively expressing tendon matrix genes, not in a dormant state.

In [ ]:
# Evidence for each label:
#   '0' Stromal:               Tppp3=0.586, Pi16=0.718, Sfrp2=0.862 — moderate of both; imprinted gene DEGs (Plagl1, Mest, H19)
#   '1' T-FAP:                 Tppp3=0.282 (lowest), Pi16=1.633, Sfrp2=1.347, Pdgfra=1.482 (all highest); complement DEGs (C3, C4b, Serping1)
#   '2' Tenogenic-progenitor:  Tppp3=-0.082 (negative), Pi16/Sfrp2 low; TrkAWT-enriched; tendon ECM DEGs (Thbs4, Kera, Col11a1)
#   '3' TSPC:                  Tppp3=1.755 / 94.5% (highest by large margin), Sfrp2=-0.005 (absent); Cd55/Ptgs1 DEGs

cluster_names = {
    '0': 'Stromal',
    '1': 'T-FAP',
    '2': 'Tenogenic-progenitor',
    '3': 'TSPC',
}

sub.obs['cell_type'] = (
    sub.obs['leiden_sub']
    .astype(str)          # cast Categorical → plain string to allow new category values
    .map(cluster_names)
    .fillna('Unknown')
)

sc.pl.umap(
    sub, color='cell_type',
    legend_loc='on data',
    title='Cluster 8 — annotated sub-clusters',
    save='_sub8_annotated.png'
)

In [ ]:
print('Cells per annotation:')
print(sub.obs['cell_type'].value_counts())
print('\nCells per annotation per condition:')
print(sub.obs.groupby(['condition', 'cell_type']).size().unstack(fill_value=0))

## 9. Save

Save the sub-clustered object and propagate the `cell_type` annotation back onto the full object for use in downstream notebooks (pySCENIC, CellRank, LIANA).

In [ ]:
sub.write('../data/Cherief_scRNA-seq/GSE244921_cluster8_sub.h5ad')
print('Saved cluster 8 sub-object.')

# Propagate annotations back to the full object
full.obs['leiden_sub'] = full.obs['leiden'].astype(str)
full.obs.loc[sub.obs_names, 'leiden_sub'] = sub.obs['leiden_sub'].values
full.obs['cell_type'] = 'Other'
full.obs.loc[sub.obs_names, 'cell_type'] = sub.obs['cell_type'].values

full.write('../data/Cherief_scRNA-seq/GSE244921_processed.h5ad')
print('Updated full object saved.')
print('\nFinal cell_type distribution in full object:')
print(full.obs['cell_type'].value_counts())